# StreamSense — TFX Training Pipeline + Kubeflow (Python 3.10, uv install)

Runs the **TFX training pipeline end-to-end** on Colab. TFX 1.15 needs Python 3.10, but Colab's
runtime is 3.12 — so instead of the heavy condacolab route (which spins up two environments and
tends to crash the session), we use **`uv`**: it fetches a standalone CPython 3.10 and installs a
**fully-pinned lock file** (`requirements-tfx-lock.txt`, generated with uv) into one venv. Nothing
to backtrack on, no kernel restart, one environment.

**What this is:** the reproducible *training* pipeline — validate → transform → train → evaluate →
push — kept separate from serving.

> Run the cells in order. **No restart happens** and there should be **no "session crashed".**
> GPU is not needed — CPU is fine for MovieLens.

## 1. Install uv + clone the repo (no restart)

In [ ]:
!pip install -q uv
!git clone https://github.com/techykajal/streamsense-ott-recsys.git
%cd streamsense-ott-recsys
!ls requirements-tfx-lock.txt          # the pinned lock must be present

## 2. Create a Python 3.10 venv and install TFX from the lock — FAST
`uv` downloads a standalone CPython 3.10 (matches the lock's build target) and installs every pinned
version directly. This is the fix for the old `resolution-too-deep` / 30-minute hang.

In [ ]:
!uv venv --seed --python 3.10 /content/tfxenv   # --seed adds pip/setuptools/wheel (TFX ships user-code via pip)
!uv pip install --python /content/tfxenv/bin/python -r requirements-tfx-lock.txt
!/content/tfxenv/bin/python -c "import tfx, tensorflow as tf; print('TFX', tfx.__version__, '| TF', tf.__version__, 'OK')" 

## 3. Prepare the pipeline's training data
Builds `data/processed/interactions.tfrecord` (the pipeline's input). Content embeddings use the
deterministic fallback (no sentence-transformers needed here).

In [ ]:
!/content/tfxenv/bin/python src/features.py

## 4. Run the TFX pipeline END-TO-END  ⭐
ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Trainer → Evaluator → Pusher.

In [ ]:
!/content/tfxenv/bin/python src/tfx_pipeline.py
print("\n=== Component artifacts produced by the pipeline ===")
!find tfx/ott_ranking -maxdepth 2 -type d 2>/dev/null | sort
print("\n=== Pushed (deployable) model ===")
!ls -R artifacts/ranking_tf 2>/dev/null | head -25 || echo "(Pusher dest = artifacts/ranking_tf in src/tfx_pipeline.py)" 

## 5. Compile the pipeline for Kubeflow (Vertex AI / GKE)

In [ ]:
!/content/tfxenv/bin/python src/tfx_pipeline.py --compile-kfp
print("=== ott_pipeline.yaml (first 45 lines) ===")
!head -45 ott_pipeline.yaml

## 6. Done — TFX pipeline executed end-to-end ✅

You ran the full DAG on real x86 Linux + Python 3.10 and compiled the Kubeflow v2 spec.

**Complete production story:**
- **Train (this notebook):** TFX validates data → trains → evaluates → **pushes** a model,
  reproducibly; compiled to a Kubeflow spec for Vertex AI / GKE.
- **Registry:** the pushed model is versioned (POC: `models/`; prod: MLflow / Vertex / S3).
- **Serve (other notebook):** TF Serving / TorchServe / Triton **load** the pushed model.

**Save to your repo:** File → Save a copy in GitHub → `techykajal/streamsense-ott-recsys`,
branch `main`, path `notebooks/StreamSense_Colab_TFX.ipynb`.

**Interview line:** "TFX is my reproducible training pipeline — validate, transform, train,
evaluate, push — compiled to a Kubeflow spec that runs on Vertex AI. Serving is decoupled and just
loads the pushed model. The install was the tricky part: TFX's dependency tree makes pip backtrack,
so I pinned it with a uv-generated lock and installed it into a uv-managed Python 3.10 for a
deterministic, fast build." 